# 04 - Detecção de Anomalias

Identifica padrões suspeitos nos fundos REAG:

1. **Anomalias de fluxo**: Z-score extremo em captação/resgate
2. **Quedas bruscas de PL**: Reduções > 20% em um dia
3. **Runs**: Sequências de resgates consecutivos
4. **Divergências**: Fluxo vs. performance

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.processors.data_processor import DataProcessor
from src.analyzers.anomaly_detector import AnomalyDetector
from config.settings import Config

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

## Carregar Dados Processados

In [ ]:
config = Config()
processor = DataProcessor(config)
detector = AnomalyDetector(config)

# Carregar dados REAG processados
data_path = config.PROCESSED_DATA_DIR / 'reag_informe_diario_processed.csv'

if data_path.exists():
    df_reag = pd.read_csv(data_path, sep=';', parse_dates=['DT_COMPTC'])
    print(f"✅ {len(df_reag):,} registros carregados")
else:
    print("⚠️ Execute primeiro o notebook 03_flow_analysis.ipynb")

## 1. Anomalias de Fluxo (Z-Score)

In [ ]:
print("🔍 Detectando anomalias de fluxo...\n")

flow_anomalies = detector.detect_flow_anomalies(df_reag, threshold=3.0)

print(f"⚠️ {len(flow_anomalies)} anomalias de fluxo detectadas\n")
print("Top 10 anomalias:")
display(flow_anomalies[[
    'CNPJ_FUNDO', 'DT_COMPTC', 'FLUXO_LIQ_DIA', 'Z_SCORE_FLOW'
]].head(10))

In [ ]:
# Visualização
if not flow_anomalies.empty:
    plt.figure(figsize=(14, 6))
    plt.scatter(
        flow_anomalies['DT_COMPTC'], 
        flow_anomalies['FLUXO_LIQ_DIA'],
        c=flow_anomalies['Z_SCORE_FLOW'].abs(),
        cmap='Reds',
        s=100,
        alpha=0.6
    )
    plt.colorbar(label='|Z-Score|')
    plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    plt.title('Anomalias de Fluxo - Fundos REAG', fontsize=14, fontweight='bold')
    plt.xlabel('Data')
    plt.ylabel('Fluxo Líquido (R$)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 2. Quedas Bruscas de PL

In [ ]:
print("🔍 Detectando quedas bruscas de PL...\n")

pl_drops = detector.detect_pl_drops(df_reag, threshold_pct=20.0)

print(f"⚠️ {len(pl_drops)} quedas bruscas de PL detectadas\n")
print("Top 10 quedas:")
display(pl_drops[[
    'CNPJ_FUNDO', 'DT_COMPTC', 'VL_PATRIM_LIQ', 'PL_VAR_PCT'
]].head(10))

## 3. Runs (Resgates Consecutivos)

In [ ]:
print("🔍 Detectando runs (resgates consecutivos)...\n")

runs = detector.detect_runs(df_reag, consecutive_days=5)

print(f"⚠️ {len(runs)} dias em runs detectados\n")

# Agrupar por fundo e run_id para contar sequências
if not runs.empty:
    run_summary = runs.groupby(['CNPJ_FUNDO', 'RUN_ID']).agg({
        'RUN_LENGTH': 'max',
        'FLUXO_LIQ_DIA': 'sum',
        'DT_COMPTC': ['min', 'max']
    }).reset_index()
    
    run_summary.columns = ['CNPJ_FUNDO', 'RUN_ID', 'DIAS_CONSECUTIVOS', 'RESGATE_TOTAL', 'DATA_INICIO', 'DATA_FIM']
    run_summary = run_summary.sort_values('DIAS_CONSECUTIVOS', ascending=False)
    
    print("Top runs por duração:")
    display(run_summary.head(10))

## 4. Divergências Fluxo vs. Performance

In [ ]:
print("🔍 Detectando divergências fluxo vs. performance...\n")

divergences = detector.detect_divergence_flow_performance(df_reag, threshold_z=2.0)

print(f"⚠️ {len(divergences)} divergências detectadas\n")
print("Top 10 divergências:")
display(divergences[[
    'CNPJ_FUNDO', 'DT_COMPTC', 'RETORNO_DIA', 'FLUXO_LIQ_DIA', 'DIVERGENCE_SCORE'
]].head(10))

## Relatório Consolidado de Anomalias

In [ ]:
print("\n" + "="*60)
print("⚠️ RELATÓRIO DE ANOMALIAS - FUNDOS REAG")
print("="*60)

print(f"\n1. Anomalias de fluxo (Z-score > 3): {len(flow_anomalies)}")
print(f"2. Quedas bruscas de PL (> 20%): {len(pl_drops)}")
print(f"3. Runs (5+ dias consecutivos): {len(run_summary) if 'run_summary' in locals() else 0}")
print(f"4. Divergências fluxo vs. performance: {len(divergences)}")

total_anomalies = len(flow_anomalies) + len(pl_drops) + len(runs) + len(divergences)
print(f"\n🔴 Total de eventos anômalos: {total_anomalies}")

## Exportar Anomalias

In [ ]:
# Salvar anomalias em arquivos separados
report_dir = config.REPORTS_DIR
report_dir.mkdir(parents=True, exist_ok=True)

if not flow_anomalies.empty:
    flow_anomalies.to_csv(report_dir / 'anomalias_fluxo.csv', index=False)
    print(f"✅ Anomalias de fluxo: {report_dir / 'anomalias_fluxo.csv'}")

if not pl_drops.empty:
    pl_drops.to_csv(report_dir / 'quedas_pl.csv', index=False)
    print(f"✅ Quedas de PL: {report_dir / 'quedas_pl.csv'}")

if not runs.empty:
    run_summary.to_csv(report_dir / 'runs_resgate.csv', index=False)
    print(f"✅ Runs: {report_dir / 'runs_resgate.csv'}")

if not divergences.empty:
    divergences.to_csv(report_dir / 'divergencias_flow_performance.csv', index=False)
    print(f"✅ Divergências: {report_dir / 'divergencias_flow_performance.csv'}")

print(f"\n📂 Relatórios salvos em: {report_dir}")

## Resumo

In [ ]:
print("\n" + "="*60)
print("✅ DETECÇÃO DE ANOMALIAS CONCLUÍDA")
print("="*60)
print(f"Total de eventos suspeitos: {total_anomalies}")
print(f"Relatórios exportados: {report_dir}")
print("\n✅ Análise completa! Próximos passos: análise manual dos relatórios.")